# Python Context Managers — `with` Statement

A **context manager** guarantees that setup and teardown code always runs — even if an exception occurs.  
The `with` statement is the cleanest way to manage resources: files, database connections, locks, timers.

```python
with some_resource as r:
    # use r here
# resource is automatically released here — no matter what
```

| Without `with` | With `with` |
|----------------|-------------|
| Must remember to close/release | Automatic cleanup |
| Resource leaks if exception occurs | Always cleaned up |
| Verbose try/finally blocks | Clean one-liner |

## 1. The Problem `with` Solves — File Handling

Without `with`, you must manually close files. An exception before `close()` causes a **resource leak**.

In [1]:
import os

# --- Old way (fragile) ---
f = open("test.txt", "w")
try:
    f.write("Hello from the old way\n")
finally:
    f.close()   # must remember this — always!

# --- The 'with' way (preferred) ---
with open("test.txt", "w") as f:
    f.write("Hello from the with statement\n")
# f.close() is called automatically here

print("File closed?", f.closed)   # True — auto-closed by 'with'

# Reading with 'with'
with open("test.txt", "r") as f:
    content = f.read()
print("Content:", content)

os.remove("test.txt")   # cleanup

File closed? True
Content: Hello from the with statement



## 2. Multiple Context Managers in One `with`

Open several resources at once — all are guaranteed to close.

In [2]:
import os

# Write two files simultaneously
with open("file_a.txt", "w") as fa, open("file_b.txt", "w") as fb:
    fa.write("Content of A\n")
    fb.write("Content of B\n")

print("Both files closed:", fa.closed, fb.closed)   # True True

# Copy file content
with open("file_a.txt", "r") as src, open("file_c.txt", "w") as dst:
    dst.write(src.read())

with open("file_c.txt") as f:
    print("Copied content:", f.read())

for fname in ["file_a.txt", "file_b.txt", "file_c.txt"]:
    os.remove(fname)

Both files closed: True True
Copied content: Content of A



## 3. How It Works — `__enter__` and `__exit__`

Any object with `__enter__` and `__exit__` methods can be a context manager.  
- `__enter__` runs when entering the `with` block (setup)  
- `__exit__` runs when leaving — **always**, even on exception (teardown)

In [3]:
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name

    def __enter__(self):
        print(f"[DB] Connecting to {self.db_name}")
        self.connected = True
        return self   # this becomes the 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"[DB] Closing connection to {self.db_name}")
        self.connected = False
        # Return False (or None) → exceptions still propagate
        # Return True → suppress the exception
        return False

    def query(self, sql):
        print(f"[DB] Running: {sql}")
        return [{"id": 1, "name": "Alice"}]


# Normal usage
print("=== Normal usage ===")
with DatabaseConnection("sales_db") as db:
    results = db.query("SELECT * FROM employees")
    print("Results:", results)
print("Connected after with:", db.connected)   # False

# Cleanup runs even when exception occurs
print("\n=== Exception inside with ===")
try:
    with DatabaseConnection("sales_db") as db:
        db.query("SELECT * FROM employees")
        raise ValueError("Something went wrong")
except ValueError as e:
    print("Caught:", e)
print("Connection was still closed:", not db.connected)

=== Normal usage ===
[DB] Connecting to sales_db
[DB] Running: SELECT * FROM employees
Results: [{'id': 1, 'name': 'Alice'}]
[DB] Closing connection to sales_db
Connected after with: False

=== Exception inside with ===
[DB] Connecting to sales_db
[DB] Running: SELECT * FROM employees
[DB] Closing connection to sales_db
Caught: Something went wrong
Connection was still closed: True


## 4. `contextlib.contextmanager` — Easier Custom Managers

Writing a class with `__enter__`/`__exit__` can be verbose. The `@contextmanager` decorator lets you write the same thing as a **generator function** — code before `yield` is setup, code after is teardown.

In [4]:
from contextlib import contextmanager
import time

# Timer context manager
@contextmanager
def timer(label=""):
    start = time.time()
    try:
        yield                   # ← 'with' block runs here
    finally:
        elapsed = time.time() - start
        print(f"{label} took {elapsed:.4f}s")

with timer("List comprehension"):
    result = [x ** 2 for x in range(100_000)]

with timer("Generator sum"):
    total = sum(x ** 2 for x in range(100_000))


# Temporary directory context manager
@contextmanager
def temp_file(path):
    import os
    print(f"Creating temp file: {path}")
    with open(path, "w") as f:
        yield f                # hand the open file to the 'with' block
    os.remove(path)
    print(f"Cleaned up: {path}")

with temp_file("temp_data.txt") as f:
    f.write("temporary content")
    print("Writing to temp file")
# file is deleted automatically after this point

List comprehension took 0.0035s
Generator sum took 0.0045s
Creating temp file: temp_data.txt
Writing to temp file
Cleaned up: temp_data.txt


## 5. Built-in Context Managers You Should Know

Python's standard library has many context managers built in.

In [5]:
import threading
from contextlib import suppress, redirect_stdout
import io

# --- threading.Lock — prevent race conditions ---
lock = threading.Lock()
counter = 0

with lock:
    counter += 1   # only one thread runs this at a time
print("Counter:", counter)

# --- contextlib.suppress — silence specific exceptions ---
# Instead of try/except just to pass:
with suppress(FileNotFoundError):
    open("does_not_exist.txt")
print("No crash — FileNotFoundError was suppressed")

# --- contextlib.redirect_stdout — capture print output ---
buffer = io.StringIO()
with redirect_stdout(buffer):
    print("This goes to the buffer, not the screen")
    print("So does this")

captured = buffer.getvalue()
print("Captured output:", repr(captured))

Counter: 1
No crash — FileNotFoundError was suppressed
Captured output: 'This goes to the buffer, not the screen\nSo does this\n'


## 6. PySpark Connection — Context Managers in Practice

PySpark's `SparkSession` should always be stopped when done. A context manager makes this automatic and safe.

In [6]:
from contextlib import contextmanager

@contextmanager
def spark_session(app_name):
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName(app_name).getOrCreate()
    try:
        yield spark
    finally:
        spark.stop()
        print(f"SparkSession '{app_name}' stopped.")

# Usage — session guaranteed to close even if an error occurs
# with spark_session("MyApp") as spark:
#     df = spark.read.csv("data.csv", header=True)
#     df.show()

print("spark_session context manager defined — ready to use with PySpark.")

spark_session context manager defined — ready to use with PySpark.


## Quick Reference

```python
# Built-in — files
with open("file.txt", "r") as f:
    data = f.read()

# Multiple resources
with open("in.txt") as src, open("out.txt", "w") as dst:
    dst.write(src.read())

# Class-based custom manager
class MyManager:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb): return False

# Decorator-based custom manager
from contextlib import contextmanager
@contextmanager
def my_manager():
    # setup
    yield resource
    # teardown (always runs)

# Suppress specific exceptions
from contextlib import suppress
with suppress(FileNotFoundError):
    os.remove("might_not_exist.txt")
```

## Interview Questions

1. What methods must a class implement to be used as a context manager?
2. What happens if an exception is raised inside a `with` block?
3. What does `__exit__` returning `True` mean?
4. How does `@contextmanager` work with `yield`?
5. What is `contextlib.suppress` and when would you use it?
6. Why is `with open(...)` preferred over `f = open(...); f.close()`?
7. Can you open two files in a single `with` statement? How?

## Practice Exercises

**Easy:**
1. Open a file called `notes.txt`, write 3 lines, close it automatically using `with`.
2. Read the same file using `with` and print each line.

**Medium:**
3. Create a `@contextmanager` called `logger` that prints `"Starting..."` before the block and `"Done."` after it.
4. Write a context manager class `TempDirectory` that creates a folder on enter and deletes it on exit.

**Advanced:**
5. Create a context manager that measures and prints elapsed time for any block of code.
6. Write a `retry` context manager that re-runs its block up to N times if a specific exception is raised.